In [96]:
"""
QQ PLOT ANALYSIS - Continuation of ETH_abnormal_returns_market_model.ipynb
================================================================================
This file generates a QQ plot to test normality of abnormal returns
from the market model already computed in the previous notebook.

Dependencies:
  - market_model_results (from previous cell)
  - plotly, statsmodels, scipy

Usage:
  - Copy this code into a new cell in your notebook after running the market model
  - All functions are self-contained
  - Output: Interactive Plotly QQ plot + normality test statistics
"""

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.graphics.gofplots import qqplot
from pathlib import Path
import pandas as pd

# Load the market model results from CSV
DATA_DIR = Path("../Data")
market_model_results = pd.read_csv(
    DATA_DIR / "market_model_results.csv",
    index_col="Date",
    parse_dates=["Date"]
)

# ============================================================================
# CELL 1: STANDARDIZATION FUNCTION
# ============================================================================

def standardize(series):
    """
    Z-score standardization of a series.
    
    Formula: z = (x - mean) / std
    
    Args:
        series: pd.Series or array-like
        
    Returns:
        pd.Series: Standardized values (mean=0, std=1)
    """
    series = pd.Series(series).dropna()
    
    if series.std() == 0:
        raise ValueError("Cannot standardize a constant series.")
    
    return (series - series.mean()) / series.std()


# ============================================================================
# CELL 2: QQ PLOT FUNCTION (PLOTLY)
# ============================================================================

def qqploting(resid, axis_range=[-5, 5]):
    """
    Create an interactive QQ plot using Plotly.
    
    The QQ plot compares:
      - X-axis: Theoretical quantiles from standard normal distribution
      - Y-axis: Sample quantiles from residuals
    
    If residuals are normally distributed, points should fall on the diagonal line.
    
    Args:
        resid: array-like, standardized residuals
        axis_range: list [min, max] for both axes (default: [-5, 5])
        
    Returns:
        plotly.graph_objects.Figure: Interactive QQ plot
    """
    # Step 1: Generate QQ plot data using statsmodels
    qqplot_data = qqplot(resid, line='s').gca().lines
    plt.close()  # Close matplotlib figure to avoid interference
    
    # Step 2: Extract coordinates from matplotlib lines
    theoretical = qqplot_data[0].get_xdata()  # Theoretical quantiles (x-axis)
    sample = qqplot_data[0].get_ydata()       # Sample quantiles (y-axis)
    
    # Step 3: Get reference line endpoints
    ref_x = qqplot_data[1].get_xdata()
    ref_y = qqplot_data[1].get_ydata()
    
    # Step 4: Create Plotly figure
    fig = make_subplots(specs=[[{"secondary_y": False}]])
    
    # Step 5: Add scatter plot (sample points - white with black border)
    fig.add_trace(go.Scatter(
        x=theoretical,
        y=sample,
        mode='markers',
        marker=dict(
            color='white',
            line=dict(color='black', width=1),
            size=6
        ),
        showlegend=False
    ), secondary_y=False)
    
    # Step 6: Add reference line (blue diagonal - y = x)
    fig.add_trace(go.Scatter(
        x=ref_x,
        y=ref_y,
        mode='lines',
        line=dict(color='blue', width=2),
        showlegend=False
    ), secondary_y=False)
    
    # Step 7: Format layout
    fig.update_layout(
        height=500,
        width=500,
        title='',
        showlegend=False,
        font=dict(family='Times New Roman', size=14),
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)',
    )
    
    # Step 8: Format axes
    fig.update_xaxes(
        title_text='Theoretical quantiles',
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True,
        showgrid=False,
        range=axis_range,
    )
    
    fig.update_yaxes(
        title_text='Sample quantiles',
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True,
        showgrid=False,
        range=axis_range,
    )
    
    fig.update_layout({'plot_bgcolor': 'rgba(0,0,0,0)',
                       'paper_bgcolor': 'rgba(0,0,0,0)'},
                      font_color='black')
    
    return fig


# ============================================================================
# CELL 3: NORMALITY TESTS FUNCTION
# ============================================================================

def perform_normality_tests(residuals):
    """
    Perform multiple normality tests on residuals.
    
    Tests included:
      - Shapiro-Wilk test
      - Kolmogorov-Smirnov test
      - Jarque-Bera test
      - Skewness and Kurtosis
    
    Args:
        residuals: array-like, residuals to test
        
    Returns:
        pd.DataFrame: Summary of test results
    """
    residuals = pd.Series(residuals).dropna()
    
    results = []
    
    # Test 1: Shapiro-Wilk (best for n < 5000)
    if len(residuals) <= 5000:
        shapiro_stat, shapiro_p = stats.shapiro(residuals)
        results.append({
            'Test': 'Shapiro-Wilk',
            'Statistic': shapiro_stat,
            'P-value': shapiro_p,
            'Result': 'Normal' if shapiro_p > 0.05 else 'Not Normal'
        })
    
    # Test 2: Kolmogorov-Smirnov
    ks_stat, ks_p = stats.kstest(residuals, 'norm', 
                                  args=(residuals.mean(), residuals.std()))
    results.append({
        'Test': 'Kolmogorov-Smirnov',
        'Statistic': ks_stat,
        'P-value': ks_p,
        'Result': 'Normal' if ks_p > 0.05 else 'Not Normal'
    })
    
    # Test 3: Jarque-Bera
    jb_stat, jb_p = stats.jarque_bera(residuals)
    results.append({
        'Test': 'Jarque-Bera',
        'Statistic': jb_stat,
        'P-value': jb_p,
        'Result': 'Normal' if jb_p > 0.05 else 'Not Normal'
    })
    
    # Test 4: Skewness and Kurtosis
    skewness = stats.skew(residuals)
    kurtosis_val = stats.kurtosis(residuals)
    results.append({
        'Test': 'Skewness',
        'Statistic': skewness,
        'P-value': np.nan,
        'Result': f'≈ 0 = {skewness:.4f}'
    })
    results.append({
        'Test': 'Excess Kurtosis',
        'Statistic': kurtosis_val,
        'P-value': np.nan,
        'Result': f'≈ 0 = {kurtosis_val:.4f}'
    })
    
    return pd.DataFrame(results)


# ============================================================================
# CELL 4: EXTRACT AND STANDARDIZE ABNORMAL RETURNS
# ============================================================================

print("="*80)
print("STEP 1: Extract Abnormal Returns from Market Model")
print("="*80)

# Extract abnormal returns from the market model results
abnormal_returns = market_model_results['abnormal_return'].copy()

print(f"\nAbnormal Returns Summary:")
print(f"  Count: {len(abnormal_returns)}")
print(f"  Mean: {abnormal_returns.mean():.6f}")
print(f"  Std: {abnormal_returns.std():.6f}")
print(f"  Min: {abnormal_returns.min():.6f}")
print(f"  Max: {abnormal_returns.max():.6f}")
print(f"  Skewness: {stats.skew(abnormal_returns):.6f}")
print(f"  Kurtosis: {stats.kurtosis(abnormal_returns):.6f}")

# Standardize abnormal returns
print("\n" + "="*80)
print("STEP 2: Standardize Abnormal Returns (Z-score)")
print("="*80)

ar_standardized = standardize(abnormal_returns)

print(f"\nStandardized Abnormal Returns Summary:")
print(f"  Count: {len(ar_standardized)}")
print(f"  Mean: {ar_standardized.mean():.6f} (should be ≈ 0)")
print(f"  Std: {ar_standardized.std():.6f} (should be ≈ 1)")
print(f"  Min: {ar_standardized.min():.6f}")
print(f"  Max: {ar_standardized.max():.6f}")


# ============================================================================
# CELL 5: CREATE QQ PLOT
# ============================================================================

print("\n" + "="*80)
print("STEP 3: Generate QQ Plot")
print("="*80)

# Create QQ plot
fig = qqploting(ar_standardized, axis_range=[-5, 5])

# Display the plot
fig.show()

# Optionally save the plot
fig.write_image("qq_plot.png", width=800, height=800, scale=2)
print("\n✓ QQ plot saved as 'qq_plot.png'")


# ============================================================================
# CELL 6: PERFORM NORMALITY TESTS
# ============================================================================

print("\n" + "="*80)
print("STEP 4: Normality Tests on Abnormal Returns")
print("="*80)

normality_results = perform_normality_tests(ar_standardized)
print("\n" + normality_results.to_string(index=False))


# ============================================================================
# CELL 7: INTERPRETATION GUIDE
# ============================================================================

print("\n" + "="*80)
print("QQ PLOT INTERPRETATION GUIDE")
print("="*80)

print("""
The QQ plot compares sample quantiles (y-axis) from abnormal returns
with theoretical quantiles (x-axis) from a standard normal distribution.

INTERPRETATION:

1. POINTS ON DIAGONAL (y = x line)
   ✓ Abnormal returns are normally distributed
   ✓ Market model assumptions satisfied
   ✓ Statistical tests (t-tests) are valid

2. S-SHAPED CURVE (concave then convex)
   ✗ Fat tails - excess kurtosis
   ✗ More extreme values than expected
   ✗ Market crashes/booms not fully captured by model

3. SYSTEMATIC CURVATURE
   ✗ Skewness present
   ✗ Distribution is asymmetric
   ✗ More extreme movements in one direction

4. DEVIATIONS AT TAILS ONLY
   ✗ Non-normality mainly in extreme events
   ✗ May need robust estimation methods
   ✗ Tail risk underestimated by normal distribution

NORMALITY TESTS INTERPRETATION (α = 0.05):

• P-value > 0.05 → Cannot reject normality (distribution is normal)
• P-value < 0.05 → Reject normality (distribution is NOT normal)

• Skewness ≈ 0 → Distribution is symmetric
• Excess Kurtosis ≈ 0 → Normal-like tails
""")


# ============================================================================
# CELL 8: SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

summary_df = pd.DataFrame({
    'Metric': [
        'Sample Size',
        'Mean (AR)',
        'Std (AR)',
        'Mean (Standardized)',
        'Std (Standardized)',
        'Skewness',
        'Kurtosis',
        'Min Value',
        'Max Value'
    ],
    'Value': [
        len(abnormal_returns),
        f"{abnormal_returns.mean():.6f}",
        f"{abnormal_returns.std():.6f}",
        f"{ar_standardized.mean():.6f}",
        f"{ar_standardized.std():.6f}",
        f"{stats.skew(abnormal_returns):.6f}",
        f"{stats.kurtosis(abnormal_returns):.6f}",
        f"{ar_standardized.min():.6f}",
        f"{ar_standardized.max():.6f}"
    ]
})

print("\n" + summary_df.to_string(index=False))

print("\n" + "="*80)
print("✓ QQ Plot Analysis Complete!")
print("="*80)

STEP 1: Extract Abnormal Returns from Market Model

Abnormal Returns Summary:
  Count: 2404
  Mean: -0.000000
  Std: 0.034061
  Min: -0.266343
  Max: 0.188809
  Skewness: -0.494203
  Kurtosis: 5.930569

STEP 2: Standardize Abnormal Returns (Z-score)

Standardized Abnormal Returns Summary:
  Count: 2404
  Mean: -0.000000 (should be ≈ 0)
  Std: 1.000000 (should be ≈ 1)
  Min: -7.819642
  Max: 5.543293

STEP 3: Generate QQ Plot



✓ QQ plot saved as 'qq_plot.png'

STEP 4: Normality Tests on Abnormal Returns

              Test   Statistic      P-value        Result
      Shapiro-Wilk    0.922057 1.409207e-33    Not Normal
Kolmogorov-Smirnov    0.093902 6.718549e-19    Not Normal
       Jarque-Bera 3620.884126 0.000000e+00    Not Normal
          Skewness   -0.494203          NaN ≈ 0 = -0.4942
   Excess Kurtosis    5.930569          NaN  ≈ 0 = 5.9306

QQ PLOT INTERPRETATION GUIDE

The QQ plot compares sample quantiles (y-axis) from abnormal returns
with theoretical quantiles (x-axis) from a standard normal distribution.

INTERPRETATION:

1. POINTS ON DIAGONAL (y = x line)
   ✓ Abnormal returns are normally distributed
   ✓ Market model assumptions satisfied
   ✓ Statistical tests (t-tests) are valid

2. S-SHAPED CURVE (concave then convex)
   ✗ Fat tails - excess kurtosis
   ✗ More extreme values than expected
   ✗ Market crashes/booms not fully captured by model

3. SYSTEMATIC CURVATURE
   ✗ Skewness present
  